# 01 · Schema Overview

**Week 1 Goal**: load every raw dataset, print shape / dtypes / null counts / sample rows, and capture findings into `docs/data_dictionary.md`.

Do **not** commit notebook outputs (handled by `nbstripout` in pre-commit, or strip manually before commit).

In [8]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

# Resolve repo root so this notebook works regardless of cwd.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'DataSet').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'DataSet'
assert DATA_DIR.exists(), f'DataSet folder not found from {Path.cwd()}'
print('Repo root:', REPO_ROOT)
print('Data dir :', DATA_DIR)
sorted(p.name for p in DATA_DIR.iterdir() if p.is_file())

Repo root: /Users/uttkarshnarayan/projects/northeastern_visualization
Data dir : /Users/uttkarshnarayan/projects/northeastern_visualization/DataSet


['ReadMe.md',
 'faculty-list-2025.xlsx',
 'grants-with-abstract.xlsx',
 'grants-with-coPI.xlsx',
 'ri_matches_grants_2026.xlsx',
 '~$faculty-list-2025.xlsx',
 '~$grants-with-coPI.xlsx',
 '~$ri_matches_grants_2026.xlsx']

In [9]:
FILES = {
    'faculty':       'faculty-list-2025.xlsx',
    'grants_abs':    'grants-with-abstract.xlsx',
    'grants_copi':   'grants-with-coPI.xlsx',
    'ri_matches':    'ri_matches_grants_2026.xlsx',
}

def load_all_sheets(path: Path) -> dict[str, pd.DataFrame]:
    """Load every sheet from an xlsx into a dict[sheet_name -> DataFrame]."""
    xl = pd.ExcelFile(path)
    return {name: xl.parse(name) for name in xl.sheet_names}

raw: dict[str, dict[str, pd.DataFrame]] = {
    key: load_all_sheets(DATA_DIR / fname) for key, fname in FILES.items()
}

# Summary of sheets per file
for key, sheets in raw.items():
    print(f'{key:12s} → {FILES[key]}')
    for sheet_name, df in sheets.items():
        print(f'    sheet={sheet_name!r:35s} shape={df.shape}')
    print()

faculty      → faculty-list-2025.xlsx
    sheet='Sheet1'                            shape=(2232, 9)

grants_abs   → grants-with-abstract.xlsx
    sheet='Grants with abstract (2).csv'      shape=(8075, 25)

grants_copi  → grants-with-coPI.xlsx
    sheet='Grants with co PI indicator (1)'   shape=(3136, 22)

ri_matches   → ri_matches_grants_2026.xlsx
    sheet='ri_matches_grants_2026-2-1_3-50'   shape=(3146, 22)



## Per-dataset profile

For each dataset we print:
1. Shape
2. Column → dtype
3. Null count per column
4. First 5 rows
5. `describe(include='all')` summary

If a file has multiple sheets we profile only the first; revisit in Week 2 if other sheets matter.

In [10]:
def profile(df: pd.DataFrame, label: str) -> None:
    print('=' * 80)
    print(f'{label}  —  shape={df.shape}')
    print('=' * 80)
    print('\n--- dtypes ---')
    print(df.dtypes.to_string())
    print('\n--- null counts (non-zero only) ---')
    nulls = df.isna().sum()
    print(nulls[nulls > 0].sort_values(ascending=False).to_string() or '(none)')
    print('\n--- head(5) ---')
    display(df.head(5))
    print('\n--- describe(include="all") ---')
    display(df.describe(include='all').T)

def first_sheet(sheets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    return next(iter(sheets.values()))

## Top 10 faculty earners by total grant dollars

Source: `ri_matches_grants_2026` (canonical structured grants table). Each grant is split across faculty rows; to avoid double-counting a multi-PI grant, we deduplicate to one `(grantid, AAUID)` row and then sum `totaldollars` per faculty member. Co-PIs share full credit for the grant they were attached to.


In [14]:
ri = first_sheet(raw['ri_matches'])

top10 = (
    ri.drop_duplicates(subset=['grantid', 'AAUID'])
      .groupby(['AAUID', 'personname'], as_index=False)
      .agg(
          total_dollars=('totaldollars', 'sum'),
          n_grants=('grantid', 'nunique'),
          n_as_copi=('iscopi', 'sum'),
          first_year=('startdateyear', 'min'),
          last_year=('startdateyear', 'max'),
      )
      .sort_values('total_dollars', ascending=False)
      .head(10)
      .reset_index(drop=True)
)
top10['total_dollars'] = top10['total_dollars'].map(lambda v: f'${v:,.0f}')
top10


,AAUID,personname,total_dollars,n_grants,n_as_copi,first_year,last_year
0,743186,"MELODIA, TOMMASO","$91,387,201",40,13,2010,2025
1,2461,"ALSHAWABKEH, AKRAM N","$87,619,945",19,5,2001,2020
2,72526,"MAKRIYANNIS, ALEXANDROS","$61,984,392",25,2,2004,2022
3,512705,"LEVINE, HERBERT","$47,122,908",36,7,2001,2025
4,270037,"LEWIS, KIM","$46,153,290",21,1,2004,2022
5,95218,"QUARANTA, VITO","$45,827,727",14,1,2004,2018
6,98196,"WINSLOW, RAI LESTER","$40,300,548",21,3,2005,2020
7,112255,"ABUR, ALI","$38,745,428",8,1,2005,2024
8,149515,"KAELI, DAVID R","$33,796,673",33,15,1995,2024
9,134929,"BRONICH, TATIANA KARPOVNA","$33,482,077",12,2,2005,2018


## Faculty Distribution

In [15]:
faculty = first_sheet(raw['faculty'])

by_college = (
    faculty.groupby('Superior_Academic_Unit', as_index=False)
           .agg(n_faculty=('Employee ID', 'nunique'))
           .sort_values('n_faculty', ascending=False)
           .reset_index(drop=True)
)
total = by_college['n_faculty'].sum()
by_college['pct_of_total'] = (by_college['n_faculty'] / total * 100).round(1).astype(str) + '%'
print(f'Total faculty: {total:,}  across {len(by_college)} colleges')
by_college


Total faculty: 2,232  across 12 colleges


,Superior_Academic_Unit,n_faculty,pct_of_total
0,College of Engineering,356,15.9%
1,College of Science,332,14.9%
2,College of Social Sciences and Humanities,304,13.6%
3,Bouvé College of Health Sciences,269,12.1%
4,D'Amore-McKim School of Business,215,9.6%
5,"College of Arts, Media and Design",210,9.4%
6,Khoury College of Computer Sciences,210,9.4%
7,Northeastern University London,126,5.6%
8,College of Professional Studies,110,4.9%
9,School of Law,58,2.6%


In [23]:
#Academic Rank
faculty = first_sheet(raw['faculty'])

by_college = (
    faculty.groupby('Academic Rank', as_index=False)
           .agg(n_faculty=('Employee ID', 'nunique'))
           .sort_values('n_faculty', ascending=False)
           .reset_index(drop=True)
)
total = by_college['n_faculty'].sum()
by_college['pct_of_total'] = (by_college['n_faculty'] / total * 100).round(1).astype(str) + '%'
print(f'Total faculty: {total:,}  across {len(by_college)} colleges')
by_college


Total faculty: 2,232  across 35 colleges


,Academic Rank,n_faculty,pct_of_total
0,Professor,433,19.4%
1,Associate Professor,268,12.0%
2,Assistant Professor,257,11.5%
3,Associate Teaching Professor,251,11.2%
4,Assistant Teaching Professor,232,10.4%
5,Teaching Professor,188,8.4%
6,Assistant Professor -UK,75,3.4%
7,Assistant Coop Coordinator,54,2.4%
8,Assistant Clinical Professor,52,2.3%
9,Associate Professor -UK,48,2.2%


In [ ]:
# College Breakdown
faculty = first_sheet(raw['faculty'])

# Pivot: Academic Unit × Academic Track Type
pivot_unit = faculty.pivot_table(
    index='Superior_Academic_Unit',
    columns='Academic Track Type',
    values='Employee ID',
    aggfunc='nunique',
    fill_value=0,
)
pivot_unit['TOTAL'] = pivot_unit.sum(axis=1)
pivot_unit = pivot_unit.sort_values('TOTAL', ascending=False)
pivot_unit.index.name = 'Academic Unit'
pivot_unit.columns.name = None

# Same column ordering: tenure-related first, then others, then TOTAL
cols = [c for c in pivot_unit.columns if c != 'TOTAL']
tenure_cols = sorted([c for c in cols if 'tenure' in str(c).lower()])
other_cols  = sorted([c for c in cols if 'tenure' not in str(c).lower()])
pivot_unit = pivot_unit[tenure_cols + other_cols + ['TOTAL']]

print(f'{len(pivot_unit)} academic units\n')
pivot_unit


12 academic units



,Non-Tenure,Tenure Track,Research Only,Teaching Only,Teaching and Research,Teaching and Scholarship,Visiting Lecturers,TOTAL
Academic Unit,,,,,,,,
College of Engineering,149,207,0,0,0,0,0,356
College of Science,143,172,0,2,2,8,5,332
College of Social Sciences and Humanities,122,152,0,0,0,4,26,304
Bouvé College of Health Sciences,174,87,0,0,0,0,8,269
D'Amore-McKim School of Business,122,93,0,0,0,0,0,215
"College of Arts, Media and Design",100,103,0,0,0,0,7,210
Khoury College of Computer Sciences,129,80,0,0,0,0,1,210
Northeastern University London,0,0,4,8,45,69,0,126
College of Professional Studies,106,1,0,0,0,0,3,110


In [22]:
# Academic Unit Breakdown
faculty = first_sheet(raw['faculty'])

# Pivot: Academic Unit × Academic Track Type
pivot_unit = faculty.pivot_table(
    index='Academic Unit',
    columns='Academic Track Type',
    values='Employee ID',
    aggfunc='nunique',
    fill_value=0,
)
pivot_unit['TOTAL'] = pivot_unit.sum(axis=1)
pivot_unit = pivot_unit.sort_values('TOTAL', ascending=False)
pivot_unit.index.name = 'Academic Unit'
pivot_unit.columns.name = None

# Same column ordering: tenure-related first, then others, then TOTAL
cols = [c for c in pivot_unit.columns if c != 'TOTAL']
tenure_cols = sorted([c for c in cols if 'tenure' in str(c).lower()])
other_cols  = sorted([c for c in cols if 'tenure' not in str(c).lower()])
pivot_unit = pivot_unit[tenure_cols + other_cols + ['TOTAL']]

print(f'{len(pivot_unit)} academic units\n')
with pd.option_context('display.max_rows', None):
    display(pivot_unit)


80 academic units



,Non-Tenure,Tenure Track,Research Only,Teaching Only,Teaching and Research,Teaching and Scholarship,Visiting Lecturers,TOTAL
Academic Unit,,,,,,,,
Khoury College of Computer Sciences,129,80,0,0,0,0,1,210
College of Professional Studies,106,1,0,0,0,0,3,110
Electrical and Computer Engineering,23,75,0,0,0,0,0,98
College of Engineering,82,0,0,0,0,0,0,82
Mechanical and Industrial Engineering,25,56,0,0,0,0,0,81
Art and Design,40,38,0,0,0,0,1,79
Physics,17,49,0,0,0,0,0,66
English,33,20,0,0,0,0,12,65
School of Law,22,33,0,0,0,0,3,58


## Action items → `docs/data_dictionary.md`

After running this notebook, copy the column lists into the data dictionary tables and annotate:
- Inferred type (after coercion)
- Nullable Y/N
- Units, value ranges, normalization rules, candidate join keys

### Notes already resolved (no advisor input needed)
- **`black` column in faculty list** — empty legacy field. Drop.
- **`grants-with-abstract` role** — text-content companion (`Title` + `Abstract`) to the structured grant tables. The 100%-null structured columns are expected; sponsor/agency info lives in the other two files. Treat as enrichment for Week 8 NLP, not a fact table.

---

### Questions for the advisor sync

Grouped from most → least time-sensitive. Each one is something I genuinely can't resolve from the data alone — either it needs institutional knowledge, a judgment call, or both.

#### A. Things that block the data model (need before Week 3)

1. **The faculty-grant join is the single biggest risk to the project.** The 2025 faculty roster uses `Employee ID` (range 26k–3.2M). The grant files use a totally different `PersonId` / `ClientFacultyId` / `AAUID` space (range 855–2.1M) and only **567 unique faculty** appear across ~2,670 grants — versus 2,232 faculty in the roster. Two questions in one:
   - Is there an authoritative crosswalk table somewhere in Provost / OPDA / NU Research that I can request?
   - If not, the realistic fallback is fuzzy-matching on `personname`. Are you OK with me accepting, say, a 90% match rate and reporting the unmatched grants as "attribution unknown"? Or is name-matching itself a non-starter for political/accuracy reasons?

2. **`grants-with-coPI` vs `ri_matches_grants_2026` — which one is the source of truth?** They are 99% the same data with snake_cased column names and 10 extra rows. I want to commit to `ri_matches` as canonical and discard the other. Any reason not to?

3. **How do I join an abstract to its grant?** The abstract file uses an `Id` space (91k–158k) that doesn't overlap with the structured `GrantId` (38k–1.8M). The most plausible bridge is `SourceActivityId` ↔ `AgencyGrantId`. Do you know how this data was originally exported? Even one example pair ("grant X has Id=91740 in abstracts and GrantId=Y in the other file") would unblock me.

4. **The grants for faculty members list all grants throughout their career** We cannot devise which grants are from their Northeastern career since we do not know when they started at Northeastern

5. **The faculty list does not account for when they were active in Northeastern** Having active time period of faculty member in Northeastern would help evaluate the grant analysis

6. **Focus on aggregrate info research first** Gradual examination of aggregrate trends - do not want to explore research ideas. 

7. **Topic analysis** What kind of research are we doing in Northeastern?

8. **Collaboration between Faculty** Review the coPI between faculty - interdisciplinary collaboration between faculty is important - interdisciplinrity between colleges in Northeastern is very relevant

9. **Correlation Matrix todo** Researchers vs coPI vs colleges.

#### B. Things that shape the narrative (need before Week 9)

4. **Who is the wider audience, really?** can mean prospective students, donors, prospective faculty, accreditation visitors, or current NU researchers — each of those wants a different story. If you had to pick the *one* visitor I should design for, who is it?

5. **What is the headline story you'd want this to tell?** I can already see candidates in the data has an explosive growth post-2010 (Khoury alone has 210 faculty now), the NSF-heavy funding mix (2,123 of ~2,670 grants), the concentration of dollars in a small group of mega-PIs (Melodia 40 grants, Levine 36, Kaeli 33). Any you would want to lead with?

6. **Are there stories the data could tell that we should *not* tell?** Like funding inequality by department, by tenure status, by gender (if we ever get that data), or year-over-year declines in specific colleges are all things this dataset can show. I'd rather know now if any of those are no-go zones, vs build a chart and have to kill it in August.

#### C. Things that affect what data I should still ask for (need before Week 6)

7. **What's missing that you wish we had?** The faculty 2025 roster - lists all faculty but does not distinguish on their current status - is this everyone who has been party of Northeastern or only active 2025 faculty members? Is there a historical faculty roster (with hire/departure dates) I should request? Same question for demographic data — I *do not* want to guess at this; want to know if it exists and whether equity analysis is in scope - or if you would want me to gather this information via scraping their linkedin or personal website information.

8. **What kind of trend are we focusing on?** Creating a time-series chart would show the "trend" on the grants coming in to Northeastern over the years. But what counts as a "trend", and what kind of "trend" do we want to focus on.

#### D. Existing knowledge I should not re-derive

9. **Has anyone done a version of this analysis before — internally, in a thesis, in an OPDA report?** I don't want to spend three weeks rediscovering something that's already in a PDF on someone's hard drive. Even a pointer to "talk to person X" would save real time.

> Skipping from this list: currency, calendar vs fiscal year, inflation deflator, `Tenure Status` nullability, and the 6-row delta between the two structured grant files. All empirically resolvable in Week 2 without advisor input.
